# Answer-first synthetic BioASQ factoid QA pilot

This notebook builds a controlled synthetic-data pilot from **training-only PubMed resources**. The manifest is already prepared deterministically:

- 250 unique source PMIDs
- two proposed questions per source (maximum 500)
- 200 training sources and 50 source-disjoint validation sources
- 125 clean sources and 125 sources assigned related distractors
- all PMIDs occurring in the real 160-question dev split or protected 95-question test split excluded

The teacher first selects exact answer spans and generates questions from a single target resource. A blinded second pass then sees the final clean or distractor evidence pack and must independently extract the same answer. Only verifier-agreed examples enter the accepted files.

The notebook defaults to **five sources** for API quality inspection. API stages cache successful responses, so the pilot can later resume across all 250 sources without repeating completed calls.


In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd

PROJECT_ROOT = Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
PYTHON = Path('/home/dinh-tuan/miniconda3/envs/bioasq/bin/python')
SCRIPT = PROJECT_ROOT / 'cse_dpo/build_synthetic_factoid_qa_pilot.py'
OUTPUT_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/synthetic_factoid_qa/gpt41mini_answer_first_250x2_v1'
MODEL = 'gpt-4.1-mini-2025-04-14'
SOURCE_LIMIT = 5  # inspect five sources first; set 0 for all 250

def run_phase(phase, *, dry_run=False):
    cmd = [
        str(PYTHON), str(SCRIPT),
        '--phase', phase,
        '--output-root', str(OUTPUT_ROOT),
        '--model', MODEL,
        '--source-limit', str(SOURCE_LIMIT),
    ]
    if dry_run:
        cmd.append('--dry-run')
    print(' '.join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

print('Output root:', OUTPUT_ROOT)


## 1. Inspect the prepared source manifest

This phase is local and has already been run. Rerunning it with the same settings deterministically recreates the manifest and makes no API calls.


In [ ]:
prepare_summary = json.loads((OUTPUT_ROOT / 'prepare_summary.json').read_text())
prepare_summary


In [ ]:
manifest = pd.read_csv(OUTPUT_ROOT / 'source_manifest_review.csv')
print(manifest.groupby(['split', 'evidence_arm']).size())
manifest.head(10)


## 2. Preview the answer-first teacher prompt

This prints one prompt and makes **no API calls**.


In [ ]:
run_phase('generate', dry_run=True)


## 3. Generate two candidate QA records per source

With `SOURCE_LIMIT = 5`, this attempts ten candidate questions. Responses are cached by the complete prompt, model, and rubric.


In [ ]:
run_phase('generate')


In [ ]:
generation_summary = json.loads((OUTPUT_ROOT / 'generation_summary.json').read_text())
generation_summary


In [ ]:
generated = []
with (OUTPUT_ROOT / 'generated_sources.jsonl').open() as handle:
    for line in handle:
        if line.strip():
            generated.append(json.loads(line))

generated_review = pd.DataFrame([
    {
        'source_id': source['synthetic_source_id'],
        'split': source['split'],
        'arm': source['evidence_arm'],
        'pmid': source['target']['pubmed_id'],
        'question_index': index,
        **item,
    }
    for source in generated
    for index, item in enumerate(source['generated_items'], 1)
])
generated_review


## 4. Preview blinded verification

The verifier is not shown the intended answers. For distractor sources, the evidence contains the target resource plus three related answer-free training resources.


In [ ]:
run_phase('verify', dry_run=True)


## 5. Run blinded verification


In [ ]:
run_phase('verify')


In [ ]:
verification_summary = json.loads((OUTPUT_ROOT / 'verification_summary.json').read_text())
verification_summary


## 6. Finalize currently verified records

Finalization makes no API calls. It requires the verifier to reproduce the intended answer under weak normalization that preserves punctuation and hyphens, cite the target resource, and confirm unique explicit support.


In [ ]:
run_phase('finalize')


In [ ]:
final_summary = json.loads((OUTPUT_ROOT / 'final_summary.json').read_text())
final_summary


In [ ]:
accepted_path = OUTPUT_ROOT / 'synthetic_accepted_all.jsonl'
accepted = [json.loads(line) for line in accepted_path.open() if line.strip()]
pd.DataFrame([{
    'id': row['synthetic_question_id'],
    'split': row['split'],
    'arm': row['evidence_arm'],
    'question': row['question'],
    'answer': row['answer'],
    'answer_type': row['answer_type'],
    'pmid': row['target_pubmed_id'],
} for row in accepted])


## 7. Review rejected or ambiguous records


In [ ]:
review = pd.read_csv(OUTPUT_ROOT / 'synthetic_review.csv')
review


## 8. Continue to all 250 sources

After reviewing the pilot, set `SOURCE_LIMIT = 0` in the settings cell and run these phases in order:

1. `generate` — up to 250 cached teacher calls, each proposing two questions.
2. `verify` — up to 250 cached blinded-verifier calls.
3. `finalize` — local filtering and export.

The resulting SFT-compatible files are:

- `synthetic_train_prepared.json`
- `synthetic_validation_prepared.json`

The real dev and test files are never modified.


In [ ]:
# After setting SOURCE_LIMIT = 0 above, uncomment and run:
# run_phase('generate')
# run_phase('verify')
# run_phase('finalize')
